### Game App Deployment

Deploys the Quest Controller Databricks App for Casper's Kitchen Rescue.
This is the mobile-first SPA that serves as the game hub.

In [ ]:
%pip install databricks-sdk --upgrade "psycopg[binary]>=3.0"

In [ ]:
import re
CATALOG = dbutils.widgets.get("CATALOG")
PROJECT_ID = re.sub(r'[^a-z0-9-]', '-', f"{CATALOG}-game".lower()).strip('-')

In [ ]:
import sys
sys.path.append('../utils')
from uc_state import add

##### Create SQL warehouse for the quest controller

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

game_warehouse_name = f"{CATALOG}-game-warehouse"

# Reuse any existing warehouse: check for the game warehouse or the default one
warehouse_id = None
for wh in w.warehouses.list():
    if wh.name and wh.name in (game_warehouse_name, f"{CATALOG}-warehouse"):
        warehouse_id = wh.id
        print(f"♻️ Reusing existing warehouse '{wh.name}' (id={wh.id})")
        break

if not warehouse_id:
    warehouse = w.warehouses.create(
        name=game_warehouse_name,
        cluster_size="2X-Small",
        max_num_clusters=1,
        min_num_clusters=1,
        enable_serverless_compute=True
    ).result()
    warehouse_id = warehouse.id
    add(CATALOG, "warehouses", warehouse)
    print(f"✅ Created warehouse (id={warehouse_id})")
else:
    print(f"✅ Using warehouse_id={warehouse_id}")

##### Write app.yaml and deploy

In [ ]:
project_name = f"projects/{PROJECT_ID}"
endpoints = list(w.postgres.list_endpoints(parent=f"{project_name}/branches/production"))
if not endpoints:
    raise RuntimeError(f"No Lakebase endpoints for {project_name}. Run Game_Lakebase first.")
ep_name = endpoints[0].name
endpoint = w.postgres.get_endpoint(name=ep_name)
lb_host = endpoint.status.hosts.host
print(f"Resolved Lakebase host: {lb_host}")

In [ ]:
from databricks.sdk.service.apps import (
    App, AppResource,
    AppResourceSqlWarehouse, AppResourceSqlWarehouseSqlWarehousePermission,
)
import os

source_code_path = os.path.abspath("../apps/quest-controller")
app_name = f"{CATALOG}-quest-controller"

desired_resources = [
    AppResource(
        name="sql-warehouse",
        sql_warehouse=AppResourceSqlWarehouse(
            id=warehouse_id,
            permission=AppResourceSqlWarehouseSqlWarehousePermission.CAN_USE
        )
    ),
]

existing_app = None
try:
    existing_app = w.apps.get(app_name)
    print(f"♻️ App '{app_name}' already exists, will update resources and redeploy")
except Exception:
    pass

if not existing_app:
    app = w.apps.create(
        App(
            name=app_name,
            default_source_code_path=source_code_path,
            resources=desired_resources,
        )
    )
else:
    try:
        w.apps.update(
            name=app_name,
            app=App(name=app_name, resources=desired_resources),
        )
        print(f"✅ Updated app resources")
    except Exception as e:
        print(f"⚠️ Could not update resources via SDK: {e}")

In [ ]:
if existing_app:
    app_status = existing_app
    print(f"♻️ Using existing app: {app_status.name}")
else:
    app_status = app.result()
    add(CATALOG, "apps", app_status)
    print(f"✅ App created: {app_status.name}")

# Ensure the app's service principal has a PostgreSQL role in Lakebase (OAuth via databricks_create_role)
import psycopg

sp_id = getattr(app_status, "service_principal_id", None)
sp_pg_name = None
if sp_id:
    try:
        sp = w.service_principals.get(sp_id)
        sp_pg_name = sp.application_id
        print(f"App SP: id={sp_id}, application_id={sp_pg_name}")
    except Exception as e:
        print(f"⚠️ Could not look up SP details: {e}")

if sp_pg_name:
    lb_cred = w.postgres.generate_database_credential(endpoint=ep_name)
    lb_user = w.current_user.me().user_name
    lb_conn = (
        f"host={lb_host} dbname=game "
        f"user={lb_user} password={lb_cred.token} sslmode=require"
    )
    try:
        with psycopg.connect(lb_conn) as conn:
            conn.autocommit = True
            with conn.cursor() as cur:
                cur.execute("CREATE EXTENSION IF NOT EXISTS databricks_auth")
                try:
                    cur.execute(f"SELECT databricks_create_role('{sp_pg_name}', 'SERVICE_PRINCIPAL')")
                    print(f"✅ Created OAuth PG role for SP: {sp_pg_name}")
                except Exception as role_err:
                    if "already exists" in str(role_err).lower():
                        print(f"♻️ OAuth PG role already exists: {sp_pg_name}")
                    else:
                        raise
                cur.execute(f'GRANT CONNECT ON DATABASE game TO "{sp_pg_name}"')
                cur.execute(f'GRANT USAGE ON SCHEMA public TO "{sp_pg_name}"')
                cur.execute(f'GRANT SELECT, INSERT, UPDATE, DELETE ON ALL TABLES IN SCHEMA public TO "{sp_pg_name}"')
                cur.execute(f'ALTER DEFAULT PRIVILEGES IN SCHEMA public GRANT SELECT, INSERT, UPDATE, DELETE ON TABLES TO "{sp_pg_name}"')
                print(f"✅ Granted table access to SP role")
    except Exception as e:
        print(f"⚠️ Could not create PG role for SP: {e}")
else:
    print("⚠️ No service_principal_id on app — Lakebase role must be created manually")

# Write app.yaml. PGUSER omitted — app uses DATABRICKS_CLIENT_ID (set automatically by Databricks Apps).
import os
import time
app_yaml_contents = f"""command:
  - uvicorn 
  - app.main:app
env:
  - name: DATABRICKS_WAREHOUSE_ID
    value: '{warehouse_id}'
  - name: DATABRICKS_CATALOG
    value: '{CATALOG}'
  - name: POSTGRES_ENDPOINT
    value: '{ep_name}'
  - name: PGHOST
    value: '{lb_host}'
  - name: PGDATABASE
    value: 'game'
  - name: PGSSLMODE
    value: 'require'
"""
yaml_path = "../apps/quest-controller/app.yaml"
if os.path.exists(yaml_path):
    os.remove(yaml_path)
time.sleep(2)
with open(yaml_path, "w") as f:
    f.write(app_yaml_contents)
print(f"✅ Wrote app.yaml (app will use DATABRICKS_CLIENT_ID for Lakebase OAuth)")

In [ ]:
# Grant app permissions
from databricks.sdk.service import catalog

# game schema needs MODIFY for INSERT/UPDATE on quest_state, leaderboard, etc.
try:
    w.grants.update(
        full_name=f"{CATALOG}.game",
        securable_type="SCHEMA",
        changes=[
            catalog.PermissionsChange(
                add=[catalog.Privilege.USE_SCHEMA, catalog.Privilege.SELECT, catalog.Privilege.MODIFY],
                principal=app_status.id
            )
        ]
    )
except Exception as e:
    print(f"\u26a0\ufe0f Grant on game: {e}")

# read-only schemas
for schema_name in ["lakeflow", "simulator"]:
    try:
        w.grants.update(
            full_name=f"{CATALOG}.{schema_name}",
            securable_type="SCHEMA",
            changes=[
                catalog.PermissionsChange(
                    add=[catalog.Privilege.USE_SCHEMA, catalog.Privilege.SELECT],
                    principal=app_status.id
                )
            ]
        )
    except Exception as e:
        print(f"\u26a0\ufe0f Grant on {schema_name}: {e}")

w.grants.update(
    full_name=f"{CATALOG}",
    securable_type="CATALOG",
    changes=[
        catalog.PermissionsChange(
            add=[catalog.Privilege.USE_CATALOG],
            principal=app_status.id
        )
    ]
)

print("\u2705 Permissions granted (game: SELECT+MODIFY, lakeflow/simulator: SELECT)")

In [ ]:
from databricks.sdk.service.apps import AppDeployment

deployment = w.apps.deploy(
    app_name=app_status.name,
    app_deployment=AppDeployment(
        source_code_path=source_code_path
    )
)
deployment_status = deployment.result()

app_url = f"{w.config.host}/apps/{app_status.name}"
print(f"\u2705 Deployed quest controller app: {app_url}")

In [ ]:
# Store app URL in game config for reference
host = w.config.host.rstrip("/")

def store_config(key, value):
    spark.sql(f"""
    MERGE INTO {CATALOG}.game.config AS target
    USING (SELECT '{key}' AS config_key, '{value}' AS config_value) AS source
    ON target.config_key = source.config_key
    WHEN MATCHED THEN UPDATE SET config_value = source.config_value
    WHEN NOT MATCHED THEN INSERT (config_key, config_value) VALUES (source.config_key, source.config_value)
    """)

store_config("quest_app_url", app_url)

# L4: Menu Knowledge Assistant — open KA configure page directly (not full list)
menu_ka_name = f"{CATALOG}-menu-knowledge"
import requests as _req
_headers = {"Authorization": f"Bearer {w.config.token}"}
try:
    from dbruntime.databricks_repl_context import get_context
    _ws_id = get_context().workspaceId
except Exception:
    _ws_id = None
try:
    _tiles_resp = _req.get(f"{host}/api/2.0/tiles", headers=_headers).json()
    _tile_id = None
    for _t in _tiles_resp.get("tiles", []):
        if _t.get("name") == menu_ka_name:
            _tile_id = _t.get("tile_id")
            break
    if _tile_id and _ws_id:
        menu_ka_url = f"{host}/ml/bricks/ka/configure/{_tile_id}?o={_ws_id}"
        print(f"\u2705 Found KA tile: {_tile_id}")
    elif _tile_id:
        menu_ka_url = f"{host}/ml/bricks/ka/configure/{_tile_id}"
        print(f"\u2705 Found KA tile: {_tile_id} (no workspace_id)")
    else:
        menu_ka_url = f"{host}/ml/agents"
        print(f"\u26A0\uFE0F KA '{menu_ka_name}' not found, falling back to Agents page")
except Exception as e:
    menu_ka_url = f"{host}/ml/agents"
    print(f"\u26A0\uFE0F Failed to look up KA tile: {e}")
store_config("menu_ka_url", menu_ka_url)
print(f"\u2705 Stored menu_ka_url: {menu_ka_url}")

menu_docs_url = f"{host}/explore/data/volumes/{CATALOG}/menu_documents/menus"
store_config("menu_docs_url", menu_docs_url)
print(f"\u2705 Stored menu_docs_url: {menu_docs_url}")

# L5: UC Lineage — Catalog Explorer URL for gold_order_header
lineage_url = f"{host}/explore/data/{CATALOG}/lakeflow/gold_order_header"
store_config("lineage_url", lineage_url)
print(f"\u2705 Stored lineage_url: {lineage_url}")

print(f"\n\U0001f3ae Quest Controller App is live!")
print(f"   URL: {app_url}")
print(f"   Open on your phone to start playing.")